In [13]:
import pandas as pd
import numpy as np

In [35]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
import joblib

In [28]:
df = pd.read_csv('../../DataSets/Churn_Modelling.csv')

In [29]:
df.dtypes

RowNumber            int64
CustomerId           int64
Surname                str
CreditScore          int64
Geography              str
Gender                 str
Age                  int64
Tenure               int64
Balance            float64
NumOfProducts        int64
HasCrCard            int64
IsActiveMember       int64
EstimatedSalary    float64
Exited               int64
dtype: object

In [30]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [31]:
Transformer_Column = ColumnTransformer(
    transformers=[
        # Drop Unnecessary Columns
        (
            "Drop Columns",
            "drop",
            ["Surname", "CustomerId", "RowNumber"]
        ),
        # Geography -> One Hot Encoding
        (
            "Encoding Geography",
            OneHotEncoder(
                drop=None,
                sparse_output=False
            ),
            ["Geography"]
        ),
        # Gender -> Ordinal Encoding
        (
            "Encoding Gender",
            OrdinalEncoder(
                categories=[["Female", "Male"]]
            ),
            ["Gender"]
        ),
        # Numerical Features -> Standard Scaling
        (
            "Scaling",
            StandardScaler(),
            [
                "CreditScore",
                "Age",
                "Tenure",
                "Balance",
                "NumOfProducts",
                "EstimatedSalary"
            ]
        )

    ],
    # Keep remaining columns unchanged
    remainder="passthrough"
)

Transformer_Column.set_output(transform="pandas")
Transformer_Column.verbose_feature_names_out = False


In [ ]:
X = df.drop("Exited", axis=1)
y = df["Exited"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [33]:
X_train = Transformer_Column.fit_transform(X_train)
X_test = Transformer_Column.transform(X_test)

In [34]:
print(Transformer_Column.get_feature_names_out())

['Geography_France' 'Geography_Germany' 'Geography_Spain' 'Gender'
 'CreditScore' 'Age' 'Tenure' 'Balance' 'NumOfProducts' 'EstimatedSalary'
 'HasCrCard' 'IsActiveMember']


In [ ]:
df_transformed_train = pd.DataFrame(X_train, columns=Transformer_Column.get_feature_names_out())
df_transformed_test = pd.DataFrame(X_test, columns=Transformer_Column.get_feature_names_out())

In [36]:
joblib.dump(Transformer_Column,"column_transformer.joblib")

['column_transformer.joblib']

In [38]:
X_train.head()

,Geography_France,Geography_Germany,Geography_Spain,Gender,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary,HasCrCard,IsActiveMember
9254,1.0,0.0,0.0,1.0,0.356500,-0.655786,0.345680,-1.218471,0.808436,1.367670,1,1
1561,0.0,1.0,0.0,1.0,-0.203898,0.294938,-0.348369,0.696838,0.808436,1.661254,1,1
1670,0.0,0.0,1.0,1.0,-0.961472,-1.416365,-0.695393,0.618629,-0.916688,-0.252807,1,0
6087,1.0,0.0,0.0,0.0,-0.940717,-1.131148,1.386753,0.953212,-0.916688,0.915393,1,0
6669,1.0,0.0,0.0,1.0,-1.397337,1.625953,1.386753,1.057449,-0.916688,-1.059600,0,0


In [39]:
X_test.head()

,Geography_France,Geography_Germany,Geography_Spain,Gender,CreditScore,Age,Tenure,Balance,NumOfProducts,EstimatedSalary,HasCrCard,IsActiveMember
6252,0.0,1.0,0.0,1.0,-0.577496,-0.655786,-0.695393,0.329937,0.808436,-1.019605,0,0
4684,1.0,0.0,0.0,1.0,-0.297297,0.390011,-1.389442,-1.218471,0.808436,0.798883,1,1
1731,0.0,0.0,1.0,0.0,-0.525607,0.485083,-0.348369,-1.218471,0.808436,-0.727980,1,0
4742,0.0,1.0,0.0,1.0,-1.511492,1.911170,1.039728,0.689272,0.808436,1.221387,1,1
4521,0.0,0.0,1.0,0.0,-0.951094,-1.131148,0.692704,0.782839,-0.916688,0.247560,1,1


In [40]:
y_train.head()

9254    0
1561    0
1670    1
6087    1
6669    1
Name: Exited, dtype: int64

In [41]:
y_test.head()

6252    0
4684    0
1731    0
4742    0
4521    0
Name: Exited, dtype: int64

In [42]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

In [43]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

/home/jayanth/miniconda3/envs/tf/lib/python3.11/site-packages/requests/__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [44]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'), 
    Dense(1, activation='sigmoid')
])

In [45]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
Optimiser = tf.keras.optimizers.Adam()
Loss = tf.keras.losses.BinaryCrossentropy()

In [49]:
model.compile(optimizer=Optimiser, loss=Loss, metrics=['accuracy'])

In [50]:
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [51]:
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [52]:
#Fitting the model with early stopping and tensorboard callback
history=model.fit(X_train, y_train, epochs=100,validation_data=(X_test, y_test),callbacks=[early_stopping, tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.8089 - loss: 0.4449 - val_accuracy: 0.8440 - val_loss: 0.3814
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8461 - loss: 0.3715 - val_accuracy: 0.8535 - val_loss: 0.3568
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8546 - loss: 0.3528 - val_accuracy: 0.8545 - val_loss: 0.3473
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8561 - loss: 0.3446 - val_accuracy: 0.8590 - val_loss: 0.3480
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8605 - loss: 0.3393 - val_accuracy: 0.8585 - val_loss: 0.3439
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8611 - loss: 0.3354 - val_accuracy: 0.8570 - val_loss: 0.3439
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8631 - loss: 0.3334 - val_accuracy: 0.8620 - val_loss: 0.3390
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8633 - loss: 0.3303 - val_acc

In [53]:
from tensorflow.keras.saving import save_model
save_model(model, 'churn_model.keras')

In [ ]:
%load_ext tensorboard

In [ ]:
%tensorboard --logdir /logs/fit/

ERROR: Timed out waiting for TensorBoard to start. It may still be running as pid 9370.